In [4]:
!pip install pandas matplotlib seaborn wordcloud

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Read Fana comments
df = pd.read_json('fana_comments.json')

# Normalize column name to 'comment' if needed
if 0 in df.columns:
    df.rename(columns={0: 'comment'}, inplace=True)
elif '0' in df.columns:
    df.rename(columns={'0': 'comment'}, inplace=True)
elif df.columns[0] != 'comment':
    df.rename(columns={df.columns[0]: 'comment'}, inplace=True)

df.head()

,comment
0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው
1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...
2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏
3,😢😢😢😢😭😭😭
4,😢😢😢😢😢


In [7]:
df.tail()

,comment
55355,الحمدلله رب العالمين 👍👍👍👍🇪🇹
55356,والله مو فاهم شيء 😁
55357,በአረቡ አለም መግኘት የነበረብንን ጎረቤት እንደመሆናችን ብዙ ነ...
55358,ዋዉ በጣም አሪፍ ነዉ
55359,በጣማ ነው ደስ የሚለው


In [8]:
df.shape

(55360, 1)

In [9]:
# Make sure column is 'comment' in case we missed it
if 'comment' not in df.columns and len(df.columns) == 1:
    df.columns = ['comment']
df = df.reset_index()
df.head()

,index,comment
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏
3,3,😢😢😢😢😭😭😭
4,4,😢😢😢😢😢


In [10]:
duplicate_count = df.duplicated().sum()
print(f"Total duplicates found: {duplicate_count}")

Total duplicates found: 0


In [11]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
index      0
comment    0
dtype: int64


In [12]:
import re

def extract_urls(text):
    return re.findall(r'https?://\S+|www\.\S+', str(text))

# Apply to dataframe
df['urls'] = df['comment'].apply(extract_urls)

# Show rows with URLs
df[df['urls'].str.len() > 0][['comment', 'urls']].head()

,comment,urls
66,https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6TaF,[https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6...
2136,https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrkF0_,[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...
2137,"( Lazy, Bankrupt, coward, Selfish) https://you...",[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...
2335,https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjcOzY,[https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjc...
3133,https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrSbZQ,[https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrS...


In [13]:
import re

# Function to remove URLs
def remove_urls(text):
    return re.sub(r'https?://[^\s]+|www\.[^\s]+', '', str(text))

# Apply cleaning
df['clean_comment'] = df['comment'].apply(remove_urls)

# Show rows where URLs existed (compare before vs after)
df[df['urls'].str.len() > 0][['comment', 'urls', 'clean_comment']].head(5)

,comment,urls,clean_comment
66,https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6TaF,[https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6...,
2136,https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrkF0_,[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...,
2137,"( Lazy, Bankrupt, coward, Selfish) https://you...",[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...,"( Lazy, Bankrupt, coward, Selfish)"
2335,https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjcOzY,[https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjc...,
3133,https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrSbZQ,[https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrS...,


In [14]:
df['has_mention'] = df['clean_comment'].str.contains(r'@\w+', na=False)

df[df['has_mention']][['clean_comment']].head(5)

,clean_comment
170,​ @mintetube391 😂
178,​ @Dagibeat21 ????
204,​ @bmsk0076 I hope we have Soon ! I wish .
266,@Dawit Getachew: Really በጉብዝናህ ወራት ጌታን አስበሃል፤ ...
711,​ @AdewaKedir 👌👌👌


In [15]:
df['has_hashtag'] = df['clean_comment'].str.contains(r'#\w+', na=False)

df[df['has_hashtag']][['clean_comment']].head(5)

,clean_comment
159,100% #1
580,amanu anteko erash tileyaleh uuuffff the poem ...
2640,▪️▫️ዘላለማዊ ድልና ክብር ለጀግናው #የኢትዬጵያ_የሀገር_ደህንነት_እና_...
2801,#27 አመት መስረቅ ስራ ነው እየተባለ በ #ህውሀት ሊቡች የተመራች #ኢት...
3019,#ፖርቲያችን ብልፅግና ባለ ብዙ ራዕይ እና ይቻላል ተብሎ የማይገመተውን ተ...


In [16]:
import re

def remove_tags(text):
    text = str(text)
    # Remove @mentions and #hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove extra whitespace left behind
    text = ' '.join(text.split())
    return text

df['clean_comment'] = df['clean_comment'].apply(remove_tags)
print("Mentions and Hashtags removed.")

Mentions and Hashtags removed.


In [17]:
df[df['clean_comment'].str.contains(r'[\d]+\s*[+\-*/=]\s*[\d]+', na=False)][['clean_comment']].head(5)

,clean_comment
482,አዎ እኛ ጴንጤ ኢየሱስ ክርስቶስ እርሱ ልክ እንደ አብ አምላክ ነው። ኢየ...
1923,E HPA Lost 4 minutes with out any Ground point...
1982,Fake election 0+0=0 wasting time and money
2898,ይታረም ኢትዮጵያ 1 - 0 ኬንያ
3383,ኢሳያስ ባለፈው ሳምንት የ80 -90 አመት አዛውንቶችን አና እድሜያቸው 1...


In [18]:
df[df['clean_comment'].str.contains(r'\d', na=False)][['clean_comment']].head(5)

,clean_comment
26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢
64,"80% of all Ethiopians life is like this, the r..."
80,ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...
98,4ኪሎ ዳይፐር ጨምሯል
106,ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ


In [19]:
punct_pattern = r"[.,!?;:\"'()\-]"

df[df['clean_comment'].str.contains(punct_pattern, na=False)][['clean_comment']].head(5)

,clean_comment
16,"May GOD send them courage ,Amen"
25,የሰራዊት ሁሉ ጌታ እግዚአብሔር መፅናናቱን ይስጣችሁ በኢየሱስ ክርስቶስ ታ...
26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢
28,በጣም ያማል! እግዚአብሔር ያጽናናችሁ!
31,እምባዬን መቆጣጠር አቃተኝ በጣም ያሳዝናል ። ፈጣሪ ያፅናችሁ !!


In [20]:
# Show comments containing numbers
print('Comments with numbers before cleaning:')
display(df[df['clean_comment'].str.contains(r'\d', na=False, regex=True)].head())

# Remove numbers from the clean_comment column
df['clean_comment'] = df['clean_comment'].str.replace(r'\d+', '', regex=True)

print('\nDataframe after removing numbers:')
display(df.head())


Comments with numbers before cleaning:


,index,comment,urls,clean_comment,has_mention,has_hashtag
26,26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢,[],እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢,False,False
64,64,"80% of all Ethiopians life is like this, the ...",[],"80% of all Ethiopians life is like this, the r...",False,False
80,80,ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...,[],ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...,False,False
98,98,4ኪሎ ዳይፐር ጨምሯል,[],4ኪሎ ዳይፐር ጨምሯል,False,False
106,106,ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ,[],ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ,False,False



Dataframe after removing numbers:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False


In [21]:
# Keep ONLY Amharic characters, emojis, and whitespace
import re

def keep_amharic_and_emoji(text):
    text = str(text)
    # Regex to NOT match Amharic, Whitespace, and Emojis ranges
    # Everything else will be replaced with ''
    pattern = r'[^\u1200-\u137F\s\U00010000-\U0010FFFF\u2600-\u27BF]'
    text = re.sub(pattern, '', text)
    # Clean up extra spaces
    return ' '.join(text.split())

print('Dataframe before keeping only Amharic & Emoji:')
display(df.head())

df['clean_comment'] = df['clean_comment'].apply(keep_amharic_and_emoji)

print('\nDataframe after keeping ONLY Amharic & Emoji:')
display(df.head())

display(df.shape)

Dataframe before keeping only Amharic & Emoji:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False



Dataframe after keeping ONLY Amharic & Emoji:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False


(55360, 6)

In [22]:
df = df[df['clean_comment'].str.strip() != '']
df = df.dropna(subset=['clean_comment'])
print(df.shape)
df.head(20)

(33136, 6)


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False
5,5,😢😢😢😢,[],😢😢😢😢,False,False
6,6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,[],የአርሲዎቹንም እገዚአብሔር ያስባቸው,False,False
7,7,ነፍስ ይማር🙏🙏,[],ነፍስ ይማር🙏🙏,False,False
8,8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,[],የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,False,False
9,9,ነፍስ የማር😢,[],ነፍስ የማር😢,False,False


In [23]:
# Comprehensive Amharic Stopwords List
amharic_stopwords = {
    "እና", "ወደ", "ከ", "ስለ", "ነው", "ነበር", "ጋር", "ደግሞ", "ብቻ", "ወዘተ", "ላይ", "ውስጥ", "በ", "እንጂ", "እንደ", "ነገር", "ግን", "ሁሉ", "አንድ", "ይህ", "ጋር", "ገና", "በኋላ",
    "ማነው", "ማን", "ምንም", "ምንድን", "ምናልባት", "እስከ", "እስከዚህ", "እስከዚያ", "እዚህ", "እዚያ", "እነዚህ", "እነዚያ", "እኔ", "አንተ", "አንቺ", "እሱ", "እሷ", "እኛ", "እናንተ", "እነሱ",
    "ናት", "ናቸው", "ነን", "ነህ", "ነሽ", "ነኝ", "ነበረ", "ነበሩ", "ነበረች", "የሆነ", "የሆኑ", "የሆነች", "ወይም", "ወይስ", "ቢሆንም", "በጣም", "አይደለም", "አይደሉም", "አይደለችም", 
    "ለ", "መሆኑን", "መሆኑ", "ማለት", "ሊሆን", "ሊሆኑ", "አሁን", "ብለዋል", "እንደነበር", "ይህም", "ብለው", "ነገሩ", "ሌሎች", "ጋር", "መሆኑን", "መሆናቸውን"
}

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in amharic_stopwords])

df['clean_comment'] = df['clean_comment'].apply(remove_stopwords)
df = df[df['clean_comment'].str.strip() != '']
print(df.shape)
df.head(10)

(33117, 6)


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False
5,5,😢😢😢😢,[],😢😢😢😢,False,False
6,6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,[],የአርሲዎቹንም እገዚአብሔር ያስባቸው,False,False
7,7,ነፍስ ይማር🙏🙏,[],ነፍስ ይማር🙏🙏,False,False
8,8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,[],የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,False,False
9,9,ነፍስ የማር😢,[],ነፍስ የማር😢,False,False


In [24]:
df['tokenized_comment'] = df['clean_comment'].str.split()
print(df.shape)
display(df.head())

(33117, 7)


,index,comment,urls,clean_comment,has_mention,has_hashtag,tokenized_comment
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False,"[እግዚአብሔር, አምላክ, ነፍሳቸውን, በአጸደ, ገነት, ያሳርፍላቸው]"
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False,"[ነፍሳችሁን, በአፀደ, ገነት, ያኑርልን, 🙏🏼ለቤተሰቦቻቸዉና, ለመላዉ, ..."
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False,"[ያሳዝናል, ፈጣሪ, መጽናናትን, ያድላቸዉ, 🙏🙏🙏]"
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False,[😢😢😢😢😭😭😭]
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False,[😢😢😢😢😢]


In [25]:
!pip install scikit-learn

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join tokens back into strings
df['text_for_tfidf'] = df['tokenized_comment'].apply(lambda x: ' '.join(x))

In [35]:
# Initialize Vectorizer
vectorizer = TfidfVectorizer()

# Create the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(df['text_for_tfidf'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)


TF-IDF Matrix Shape: (33117, 75384)


In [39]:
# Get the feature names
feature_names = vectorizer.get_feature_names_out()

# Show 50 words from the middle of the list (where actual Amharic words are)
print(feature_names[1000:1050])

['ህገወጥነት' 'ህገወጥነትለነሱይሰራ' 'ህጉ' 'ህጉማ' 'ህጉንም' 'ህጋችን' 'ህጋዊ' 'ህጋዊውን' 'ህግ' 'ህግም'
 'ህግበፀረ' 'ህግና' 'ህግን' 'ህግንም' 'ህግንና' 'ህግደፍ' 'ህግደፍን' 'ህጎ' 'ህጎች' 'ህጎን' 'ህጭብ'
 'ህጻኑ' 'ህጻናት' 'ህጻናትና' 'ህጻናትን' 'ህጻን' 'ህፃነት' 'ህፃኑ' 'ህፃና' 'ህፃናት' 'ህፃናትም'
 'ህፃናትን' 'ህፃን' 'ህፃውንቲ' 'ህፈረት' 'ህፍ' 'ህፍረት' 'ሆሆ' 'ሆሆሆ' 'ሆሆሆሆ' 'ሆሆሆሆይይይ'
 'ሆለታ' 'ሆላ' 'ሆልደር' 'ሆመቾ' 'ሆሚቾ' 'ሆምቾ' 'ሆርማት' 'ሆሮ' 'ሆሳዕና']


In [ ]:
# # Define your Amharic sentiment words
# pos_words = ["ጥሩ", "አሪፍ", "ምርጥ", "ደስ", "አሜን", "በርታ", "ትክክል", "ሰላም"]
# neg_words = ["መጥፎ", "ክፉ", "ውሸት", "ሌባ", "ሞት", "ሀዘን", "ጦርነት", "አይረባም"]

# # Get the feature names (words) from the vectorizer
# feature_names = vectorizer.get_feature_names_out().tolist()

# # Find the index position of our sentiment words in the TF-IDF matrix
# pos_indices = [feature_names.index(w) for w in pos_words if w in feature_names]
# neg_indices = [feature_names.index(w) for w in neg_words if w in feature_names]

In [40]:
# 1. Expanded Lexicon including Emojis
pos_words = [
    "ጥሩ", "አሪፍ", "ምርጥ", "ደስ", "አሜን", "በርታ", "ትክክል", "ሰላም", "እግዚአብሔር", "ፈጣሪ", "ይባርክ", 
    "❤️", "🙌", "🔥", "👏", "🙏", "✅", "🥰", "💪"
]

neg_words = [
    "መጥፎ", "ክፉ", "ውሸት", "ሌባ", "ሞት", "ሀዘን", "ጦርነት", "አይረባም", "ሞኝ", "ያሳዝናል", "ወንጀል",
    "😭", "😢", "☹️", "💔", "😡", "👎", "🤮", "💀"
]

# 2. Get the feature names from the vectorizer
feature_names = vectorizer.get_feature_names_out().tolist()

# 3. Find index positions (using a safer method)
pos_indices = [i for i, word in enumerate(feature_names) if word in pos_words]
neg_indices = [i for i, word in enumerate(feature_names) if word in neg_words]

print(f"Found {len(pos_indices)} positive and {len(neg_indices)} negative sentiment markers.")

Found 11 positive and 11 negative sentiment markers.


In [37]:
import numpy as np

def calculate_label(row_index):
    # Get weights for the current row
    row_data = tfidf_matrix.getrow(row_index)
    
    # Sum TF-IDF weights for positive and negative words found in this row
    pos_score = sum(row_data[0, idx] for idx in pos_indices)
    neg_score = sum(row_data[0, idx] for idx in neg_indices)
    
    if pos_score > neg_score:
        return 'Positive'
    elif neg_score > pos_score:
        return 'Negative'
    else:
        return 'Neutral'

# Apply the labeling to every row
df['sentiment'] = [calculate_label(i) for i in range(df.shape[0])]

In [38]:
print("Sentiment Distribution:")
print(df['sentiment'].value_counts())

# Show the first few rows with the new label
display(df[['clean_comment', 'sentiment']].head(10))

Sentiment Distribution:
sentiment
Neutral     30642
Positive     2081
Negative      394
Name: count, dtype: int64


,clean_comment,sentiment
0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,Neutral
1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,Neutral
2,ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,Neutral
3,😢😢😢😢😭😭😭,Neutral
4,😢😢😢😢😢,Neutral
5,😢😢😢😢,Neutral
6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,Neutral
7,ነፍስ ይማር🙏🙏,Neutral
8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,Positive
9,ነፍስ የማር😢,Neutral
